<a href="https://colab.research.google.com/github/long7872/GNN/blob/main/Chapter%207/Section_7_10_Graph_Coarsening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chapter 8 Learning at Scale - Section 8.8 - Graph Coarsening

-------------------------------------------------------------------

This script demonstrates the process of graph coarsening using the Graclus
method with PyTorch Geometric. It starts with loading the KarateClub dataset,
a well-known social network dataset representing the relationships between
34 members of a karate club at a US university in the 1970s.

The goal is to perform graph coarsening to reduce the graph's complexity while
maintaining its essential structural and feature properties. The Graclus
method is used for clustering, followed by max-pooling to create a coarsened
graph. Node labels are also updated to reflect the most common label within
each cluster.

Our process below:
1. The KarateClub dataset is loaded.
2. Graclus clustering is performed on the graph.
3. New labels are generated for the coarsened graph.
4. An original training mask is assumed; if it doesn't exist, all nodes are considered for training.
5. A new training mask for the coarsened graph is created based on whether the clusters contain any training nodes from the original graph.
6. The new training mask is added to the coarsened data object.
7. The coarsened data object, with the new training mask, is printed for verification.

# Part I. Install Packages, Load Data and Import Packages

In [1]:
!pip install torch==2.11.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 247.2 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 231.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 238.2 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 MB 258.5 MB/s  0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.1/531.1 MB 32.5 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 53.2 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 95.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 64.4 MB/s  0:00:00
  Attempting uninstall: nvidia-cusparselt-cu13
    Found existing installation: nvidia-cusparselt-cu13 0.8.1
    Uninstalling nvidia-cusparselt-cu13-0.8.1:
      S

In [2]:
# Find the CUDA version PyTorch was installed with
!python -c "import torch; print(torch.version.cuda)"

13.0


In [3]:
# PyTorch version
!python -c "import torch; print(torch.__version__)"

2.11.0+cu130


In [4]:
# Use the above information to fill in the http address below
# %%capture
# !pip install ogb pyg-lib torch-scatter torch-cluster torch-sparse -f https://data.pyg.org/whl/torch-2.0.1+cu118.html
!pip install torch-geometric
!pip install ogb pyg-lib torch-scatter torch-cluster torch-sparse -f https://data.pyg.org/whl/torch-2.11.0+cu130.html
# !pip install -q "pyg_lib>=0.6.0" -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.0 MB/s  0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 105.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 132.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 112.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [ogb]


In [6]:
import torch
from torch_geometric.nn import graclus, max_pool
from torch_geometric.utils import to_undirected
from torch_geometric.datasets import KarateClub
from scipy import stats


##Coarsen the karate club dataset

In [7]:

# Load dataset
dataset = KarateClub()
data = dataset[0]

# Print the original labels
print("Old Labels:")
print(data.y)

# Ensure edge_index is undirected for Graclus clustering
data.edge_index = to_undirected(data.edge_index)

"""
Perform clustering using the Graclus method. The Graclus algorithm is a
fast graph clustering method that maximizes the modularity over possible
clusterings. This section calculates clusters and prints them.
"""
cluster = graclus(data.edge_index, num_nodes=data.num_nodes)
cluster_n = cluster.to('cpu').numpy()
y_n = data.y.to('cpu').numpy()

"""
Generate new labels for the coarsened graph. For each cluster generated
by Graclus, find the most common label among the nodes in the cluster
and assign it as the new label for the coarsened node corresponding to
that cluster.
"""
new_labels = []
for i in range(cluster_n.max() + 1):
    labels_in_cluster = y_n[cluster_n == i]
    if labels_in_cluster.size > 0:
        most_common_label = stats.mode(labels_in_cluster).mode.item()
        new_labels.append(most_common_label)

# Convert new_labels to a tensor
new_labels = torch.tensor(new_labels, dtype=torch.long)

# Print the new labels
print("\nNew Labels:")
print(new_labels)

# Print the old data object
print("\nOld Data Object:")
print(data)

"""
Perform max pooling to obtain the coarsened graph, using the clusters
obtained from the Graclus algorithm. Max pooling helps to reduce the
graph size while retaining the essential structural and feature
characteristics.
"""
data_coarse = max_pool(cluster, data)
data_coarse.y = new_labels

# Assuming an original training mask exists; if not, consider all nodes for training
original_train_mask = data.train_mask if hasattr(data, 'train_mask') else torch.ones(data.num_nodes, dtype=bool)

# Print the original training mask
print("\nOriginal Training Mask:")
print(original_train_mask)

# Determine whether clusters contain any training nodes from the original graph
contains_training_node = torch.zeros(cluster_n.max() + 1, dtype=bool)
for i in range(cluster_n.max() + 1):
    nodes_in_cluster = cluster_n == i
    if original_train_mask[nodes_in_cluster].any().item():
        contains_training_node[i] = True

# Create a new training mask for the coarsened graph
new_train_mask = contains_training_node[new_labels.long()]

# Print the new training mask
print("\nNew Training Mask:")
print(new_train_mask)

# Add the new training mask to the coarsened data object
data_coarse.train_mask = new_train_mask

# Print the new coarsened data object with the training mask
print("\nNew Data Object with Training Mask:")
print(data_coarse)


Old Labels:
tensor([1, 1, 1, 1, 3, 3, 3, 1, 0, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0,
        2, 2, 0, 0, 2, 0, 0, 2, 0, 0])

New Labels:
tensor([1, 1, 1, 1, 3, 3, 1, 0, 1, 1, 0, 0, 3, 1, 0, 0, 1, 0, 0, 2, 0, 0, 2, 0])

Old Data Object:
Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])

Original Training Mask:
tensor([ True, False, False, False,  True, False, False, False,  True, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False,  True, False, False, False, False, False,
        False, False, False, False])

New Training Mask:
tensor([False, False, False, False, False, False, False,  True, False, False,
         True,  True, False, False,  True,  True, False,  True,  True, False,
         True,  True, False,  True])

New Data Object with Training Mask:
DataBatch(x=[24, 34], edge_index=[2, 106], y=[24], train_mask=[24])


##Coarsen the Amazon products dataset

In [8]:
# importing obg datatset
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.transforms as T

In [9]:
import os.path as osp
root = osp.join(osp.dirname(osp.realpath('./')), 'data', 'products')


In [14]:
!apt-get install -qq aria2
!mkdir -p dataset
!aria2c -x 16 -s 16 -k 1M -d dataset -o products.zip http://snap.stanford.edu/ogb/data/nodeproppred/products.zip
!unzip -q -o dataset/products.zip -d dataset/

Selecting previously unselected package libcares2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a sym

In [15]:
# Note: If you encounter some strange error, just run this cell again.
import builtins
import torch
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage
from torch_geometric.data import Data, DataLoader
from ogb.nodeproppred import PygNodePropPredDataset

torch.serialization.add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])

_orig_input = builtins.input
builtins.input = lambda *args: 'y'

try: dataset_dense = PygNodePropPredDataset(name='ogbn-products')
finally: builtins.input = _orig_input

dataset_dense.num_classes

ogbn-products has been updated.
Using exist file products.zip
Extracting dataset/products.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:01<00:00,  1.06s/it]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 2692.11it/s]


Saving...


Done!


47

In [17]:
%%time
from torch_geometric.nn.pool.consecutive import consecutive_cluster
# Load dataset
dataset = dataset_dense
data = dataset[0]

# Print the original labels
print("Old Labels:")
print(data.y)

# Ensure edge_index is undirected for Graclus clustering
# data.edge_index = to_undirected(data.edge_index)
# Already undirected

"""
Perform clustering using the Graclus method. The Graclus algorithm is a
fast graph clustering method that maximizes the modularity over possible
clusterings. This section calculates clusters and prints them.
"""
cluster = graclus(data.edge_index, num_nodes=data.num_nodes)
# cluster_n = cluster.to('cpu').numpy()
# y_n = data.y.to('cpu').numpy()
cluster_consecutive, perm = consecutive_cluster(cluster)
num_clusters = perm.size(0)

"""
Generate new labels for the coarsened graph. For each cluster generated
by Graclus, find the most common label among the nodes in the cluster
and assign it as the new label for the coarsened node corresponding to
that cluster.
"""
# new_labels = []
# for i in range(cluster_n.max() + 1):
#     labels_in_cluster = y_n[cluster_n == i]
#     if labels_in_cluster.size > 0:
#         most_common_label = stats.mode(labels_in_cluster).mode.item()
#         new_labels.append(most_common_label)

# # Convert new_labels to a tensor
# new_labels = torch.tensor(new_labels, dtype=torch.long)

y_1d = data.y.squeeze(-1)
new_labels = torch.full((num_clusters,), y_1d.max().item() + 1, dtype=torch.long, device=y_1d.device)
new_labels.scatter_reduce_(0, cluster_consecutive, y_1d.long(), reduce='amin', include_self=False)

# Print the new labels
print("\nNew Labels:")
print(new_labels)

# Print the old data object
print("\nOld Data Object:")
print(data)

"""
Perform max pooling to obtain the coarsened graph, using the clusters
obtained from the Graclus algorithm. Max pooling helps to reduce the
graph size while retaining the essential structural and feature
characteristics.
"""
data_coarse = max_pool(cluster_consecutive, data)
data_coarse.y = new_labels

# Assuming an original training mask exists; if not, consider all nodes for training
original_train_mask = data.train_mask if hasattr(data, 'train_mask') else torch.ones(data.num_nodes, dtype=bool)

# Print the original training mask
print("\nOriginal Training Mask:")
print(original_train_mask)

# Determine whether clusters contain any training nodes from the original graph
# contains_training_node = torch.zeros(cluster_n.max() + 1, dtype=bool)
# for i in range(cluster_n.max() + 1):
#     nodes_in_cluster = cluster_n == i
#     if original_train_mask[nodes_in_cluster].any().item():
#         contains_training_node[i] = True
contains_training_node = torch.zeros(num_clusters, dtype=torch.int8, device=cluster.device)
contains_training_node.scatter_reduce_(0, cluster_consecutive, original_train_mask.to(torch.int8), reduce='amax', include_self=False)

# Create a new training mask for the coarsened graph
new_train_mask = contains_training_node.bool()

# Print the new training mask
print("\nNew Training Mask:")
print(new_train_mask)

# Add the new training mask to the coarsened data object
data_coarse.train_mask = new_train_mask

# Print the new coarsened data object with the training mask
print("\nNew Data Object with Training Mask:")
print(data_coarse)


Old Labels:
tensor([[0],
        [1],
        [2],
        ...,
        [8],
        [2],
        [4]])

New Labels:
tensor([ 0,  1,  2,  ..., 42,  8,  4])

Old Data Object:
Data(num_nodes=2449029, edge_index=[2, 123718280], x=[2449029, 100], y=[2449029, 1])

Original Training Mask:
tensor([True, True, True,  ..., True, True, True])

New Training Mask:
tensor([True, True, True,  ..., True, True, True])

New Data Object with Training Mask:
DataBatch(x=[1432409, 100], edge_index=[2, 98508852], y=[1432409], train_mask=[1432409])
CPU times: user 58.6 s, sys: 10.3 s, total: 1min 8s
Wall time: 15.2 s
